In [1]:
import os

In [2]:
%pwd

'c:\\Users\\lenovo\\Desktop\\Kidney-Disease-Prediction\\research'

In [3]:
os.chdir("..")

In [4]:
%pwd

'c:\\Users\\lenovo\\Desktop\\Kidney-Disease-Prediction'

In [ ]:
mlflow_uri="https://dagshub.com/likhisgowda2005/Kidney-Disease-Prediction.mlflow"

# mlflow_uri="https://dagshub.com/likhisgowda2005/Kidney-Disease-Prediction.mlflow/#/experiments"

In [6]:
dags_token = "407996f462ba4359ac62d5444ca0d387509d380c"

In [7]:
import os
os.environ['MLFLOW_TRACKING_URI'] = "https://dagshub.com/likhisgowda2005/Kidney-Disease-Prediction.mlflow"
os.environ['MLFLOW_TRACKING_USERNAME'] = "likhisgowda2005"
os.environ['MLFLOW_TRACKING_PASSWORD'] = "407996f462ba4359ac62d5444ca0d387509d380c"  # Use your DagsHub token

In [8]:
import tensorflow as tf

In [9]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

In [10]:
from dataclasses import dataclass
from pathlib import Path
@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int
    

In [11]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml,create_directories,save_json

In [12]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
        
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model = "artifacts/training/model.h5",
            training_data = "artifacts/data_ingestion/Chest CT Scan data",
            all_params = self.params,
            mlflow_uri = "https://dagshub.com/likhisgowda2005/Kidney-Disease-Prediction.mlflow",
            params_image_size = self.params.IMAGE_SIZE,
            params_batch_size = self.params.BATCH_SIZE
        )
        return eval_config

In [13]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

c:\Users\lenovo\Desktop\GenAI\Python\kidneyCnn\lib\site-packages\mlflow\utils\requirements_utils.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [14]:
class Evaluation:
    def __init__(self,config:EvaluationConfig):
        self.config = config
        
    def _valid_generator(self):
        
        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split = 0.30
        )
        
        dataflow_kwargs = dict(
            target_size = self.config.params_image_size[:-1],
            batch_size = self.config.params_batch_size,
            interpolation = "bilinear"
        )
        
        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )
        
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory = self.config.training_data,
            subset = "validation",
            shuffle = False,
            **dataflow_kwargs
        )
    
    @staticmethod
    def load_model(path:Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    
    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)
        self.save_score()
        
    def save_score(self):
        scores = {"loss":self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"),data=scores)
        
    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss":self.score[0],"accuracy": self.score[1]}
            )
            # Model registry does not work with file store
            
            if tracking_url_type_store != "file":
                
                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://www.mlflow.org/docs/latest/ml/model-registry/
                mlflow.keras.log_model(self.model,"model",registered_model_name="VGG16Model")
            else:
                mlflow.keras.log_model(self.model,"model")
    

In [15]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(config=eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()
    # print(f"Evaluation score:{evaluate.score}")
except Exception as e:
    raise e

[2026-02-01 07:59:06,376: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-02-01 07:59:06,392: INFO: common: yaml file: params.yaml loaded successfully]
[2026-02-01 07:59:06,397: INFO: common: created directory at: artifacts]
Found 2207 images belonging to 2 classes.
138/138 [==============================] - 830s 6s/step - loss: 18.4828 - accuracy: 0.6901
[2026-02-01 08:12:59,037: INFO: common: json file saved at: scores.json]


2026/02/01 08:13:03 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


[2026-02-01 08:13:05,768: WARNING: save: Found untraced functions such as _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op while saving (showing 5 of 14). These functions will not be directly callable after loading.]
INFO:tensorflow:Assets written to: C:\Users\lenovo\AppData\Local\Temp\tmpixpvb8ln\model\data\model\assets
[2026-02-01 08:13:07,760: INFO: builder_impl: Assets written to: C:\Users\lenovo\AppData\Local\Temp\tmpixpvb8ln\model\data\model\assets]


2026/02/01 08:13:25 WARNING mlflow.utils.requirements_utils: The following packages were not found in the public PyPI package index as of 2023-02-28; if these packages are not present in the public PyPI index, you must install them manually before loading your model: {'backports-tarfile'}
c:\Users\lenovo\Desktop\GenAI\Python\kidneyCnn\lib\site-packages\_distutils_hack\__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
Successfully registered model 'VGG16Model'.
2026/02/01 08:22:30 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: VGG16Model, version 1
Created version '1' of model 'VGG16Model'.


In [20]:
import dagshub
dagshub.init(repo_owner='likhisgowda2005', repo_name='Kidney-Disease-Prediction', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

ModuleNotFoundError: No module named 'dagshub'